# Mortality estimation model training

Here we will begin the mortality estimation model training (with binary data) for which we first need to establish the dataset for it to be used in tensorflow.

## Preparing the dataset

### Importing necessary libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import tensorflow as tf
from tensorflow.keras import Model, layers

from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import keras
import os

2025-06-24 11:48:58.253615: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-24 11:48:58.472798: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-24 11:48:58.474719: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-24 11:49:01.848396: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


### Setting parameters

In [ ]:
# Data parameters
im_width = 256
im_height = im_width
n_classes = 5

# Training Parameters
learning_rate = 0.001
epochs = 30
batch_size = 32
s_per_epoch = 20
last_dense = 32
print(f"Batch size: {batch_size}")
print(f"Units of the last Dense Layer: {last_dense}")

### Reading the data

In [ ]:
features_folders = sorted(os.listdir("patrones_binary/"))[:-1]

# First iteration through folders to know total number of pathces
patchs_total_number = 0
for f_folder in features_folders:
    patchs_total_number += len(
        next(os.walk("patrones_binary/%s/" % (f_folder)))[2]
    )  # list of names all images in the given path

images = np.zeros((patchs_total_number, im_height, im_width), dtype=np.float32)
labels = np.zeros((patchs_total_number), dtype=np.float32)

n = 0
for folder in features_folders:
    names = next(os.walk("patrones_binary/%s/" % (folder)))[2]  # list of names all images in the given path

    for name in names:
        # Load features
        feature = np.loadtxt("patrones_binary/%s/%s" % (folder, name))

        # Load labels
        namesplit = name.split(sep="_")
        label = namesplit[1]  # omega
        
        images[n] = feature
        labels[n] = label

        n += 1

### Separating in train/test/validation

In [ ]:
images_trainval, images_test, labels_trainval, labels_test = train_test_split(
    images[:, :, :],
    labels,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

images_train, images_val, labels_train, labels_val = train_test_split(
    images_trainval[:, :, :],
    labels_trainval,
    test_size=(1/8),
    random_state=42,
    shuffle=True,
)

[=================================================================================================== ] 99%

### Model training

In [ ]:
# Define model (https://www.tensorflow.org/tutorials/images/cnn?hl=es-419)

model = tf.keras.models.Sequential([
  tf.keras.layers.Conv2D(filters = 32, kernel_size = (3, 3), activation='relu', input_shape=(256, 256, 1)),
  tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
  tf.keras.layers.Conv2D(filters = 32, kernel_size = (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
  tf.keras.layers.Conv2D(filters = 32, kernel_size = (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
  tf.keras.layers.Conv2D(filters = 32, kernel_size = (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
  tf.keras.layers.Conv2D(filters = 32, kernel_size = (3, 3), activation='relu'),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(units = last_dense, activation='relu'),
  tf.keras.layers.Dense(1)
])

2025-06-20 11:24:51.152669: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [ ]:
# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[tf.keras.metrics.MeanAbsoluteError(),
             tf.keras.metrics.MeanSquaredLogarithmicError(),
             ],
)

In [ ]:
model.summary()

In [ ]:
# Train model
result = model.fit(
        x=images_train,
        y=labels_train,
        validation_data=(images_val, labels_val),
        batch_size=batch_size,
        epochs=epochs,
        steps_per_epoch=s_per_epoch
    )


In [ ]:
i = 1
model_path = f"model_ME_{i}.h5"
model.save(model_path)

In [ ]:
# Print results
loss_train, MAE_train, MSLE_train = model.evaluate(images_train, labels_train, batch_size=1000)

print(f"Loss train: {loss_train}")
print(f"MAE: {MAE_train}")
print(f"MSLE: {MSLE_train}")


loss_test, MAE_test, MSLE_test = model.evaluate(images_test, labels_test, batch_size=1000)

print(f"Loss test: {loss_test}")
print(f"MAE: {MAE_test}")
print(f"MSLE: {MSLE_test}")

In [ ]:
plt.rcParams.update({'font.size': 14})

y_score_train = model.predict(images_train)
plt.scatter(labels_train, y_score_train, marker='.', s=10, label = f'{i}')

plt.ylabel('Predictions')
plt.xlabel('Labels')
plt.grid(visible=True)
plt.savefig("Predictions_modelB.pdf", format="pdf", bbox_inches="tight") 
plt.show()

In [ ]:
plt.rcParams.update({'font.size': 14})

y_score_test = model.predict(images_test)
plt.scatter(labels_test, y_score_test, marker='.', s=10, label = f'{i}')

plt.ylabel('Predictions')
plt.xlabel('Labels')
plt.grid(visible=True)
plt.savefig("Predictions_test_model_ME.pdf", format="pdf", bbox_inches="tight") 
plt.show()

In [ ]:
i = 1
results = pd.DataFrame(result.history)

results.to_csv(f'results_model_ME_{i}.csv', index=False)